# 12.7 深度研究智能体 (Deep Research Agents)

> 🕐 预估学习时间：40分钟

深度研究 Agent（如 OpenAI Deep Research、各类 survey agent）把问题拆解为多轮检索-阅读-综合-引用，强调证据链与可追溯，而不是单次 RAG。

本节涵盖：
- 研究状态机
- 子问题分解与检索循环
- 证据库与冲突检测
- 带引用的报告合成
- 预算控制与停止条件


## 1. 研究状态机

`Plan → Search → Read → Note → Synthesize → Critique → (loop|finish)`

每步消耗预算（检索次数/token），需显式停止条件。


In [ ]:
from dataclasses import dataclass, field
from typing import Any
import hashlib

torch = __import__('torch')  # keep torch import style consistent if needed later


@dataclass
class Evidence:
    eid: str
    query: str
    source: str
    snippet: str
    score: float


@dataclass
class ResearchState:
    question: str
    subquestions: list[str] = field(default_factory=list)
    evidence: list[Evidence] = field(default_factory=list)
    notes: list[str] = field(default_factory=list)
    budget_search: int = 6
    done: bool = False


def plan(question: str) -> list[str]:
    # toy decomposition
    seeds = ['definition', 'methods', 'benchmarks', 'open problems']
    return [f'{question}: {s}' for s in seeds]


state = ResearchState('LLM inference optimization')
state.subquestions = plan(state.question)
print('=== Research Plan ===')
for i, sq in enumerate(state.subquestions, 1):
    print(f'{i}. {sq}')
print(f'Key: Explicit subquestions turn vague research into measurable retrieval goals.')


## 2. 检索-阅读-记笔记循环

对每个子问题检索 Top-K，提炼笔记，写入证据库；重复直到预算耗尽或信息增益低。


In [ ]:
CORPUS = {
    'p1': 'PagedAttention manages KV cache like virtual memory to reduce fragmentation.',
    'p2': 'Speculative decoding uses a draft model to propose tokens verified by the target.',
    'p3': 'Prefill is compute-bound while decode is memory-bandwidth-bound.',
    'p4': 'Continuous batching improves GPU utilization for online serving.',
    'p5': 'Quantization (GPTQ/AWQ) reduces memory footprint with small quality loss.',
}


def search(query: str, k=2) -> list[Evidence]:
    scored = []
    q_terms = set(query.lower().split())
    for sid, text in CORPUS.items():
        terms = set(text.lower().replace(',', ' ').split())
        score = len(q_terms & terms) + 0.1 * text.lower().count('kv')
        scored.append((score, sid, text))
    scored.sort(reverse=True)
    out = []
    for score, sid, text in scored[:k]:
        eid = hashlib.md5(f'{sid}:{query}'.encode()).hexdigest()[:8]
        out.append(Evidence(eid, query, sid, text, float(score)))
    return out


def note_from_evidence(ev: Evidence) -> str:
    return f'[{ev.source}] {ev.snippet}'


print('=== Search-Read Loop ===')
for sq in state.subquestions:
    if state.budget_search <= 0:
        break
    hits = search(sq, k=2)
    state.budget_search -= 1
    for ev in hits:
        if all(e.snippet != ev.snippet for e in state.evidence):
            state.evidence.append(ev)
            state.notes.append(note_from_evidence(ev))
    print(f'Q: {sq} -> {[h.source for h in hits]} budget={state.budget_search}')
print(f'evidence={len(state.evidence)} notes={len(state.notes)}')
print(f'\nKey: Deduplicate evidence and spend budget on uncovered subquestions first.')


## 3. 冲突检测与综合成文

若两条证据结论冲突，标记不确定性并建议追加检索；最终答案强制引用证据 ID。


In [ ]:
def detect_conflicts(evidence: list[Evidence]) -> list[tuple[str, str]]:
    # toy: treat presence of opposing keywords as conflict
    pos = [e for e in evidence if 'improves' in e.snippet or 'reduce' in e.snippet]
    neg = [e for e in evidence if 'loss' in e.snippet and 'quality' in e.snippet]
    conflicts = []
    if pos and neg:
        conflicts.append((pos[0].eid, neg[0].eid))
    return conflicts


def synthesize(question: str, evidence: list[Evidence]) -> str:
    bullets = []
    for e in evidence[:5]:
        bullets.append(f'- {e.snippet} [{e.eid}]')
    body = '\n'.join(bullets)
    return f'# Report: {question}\n\n## Findings\n{body}\n'


conflicts = detect_conflicts(state.evidence)
report = synthesize(state.question, state.evidence)
print('=== Synthesis ===')
print('conflicts:', conflicts)
print(report)
print(f'Key: Citations make deep research auditable; conflicts trigger more search instead of fluent hallucination.')


## 4. 停止条件与产业要点

停止当：
1. 所有子问题都有 ≥1 条证据  
2. 新增检索的信息增益低于阈值  
3. 预算耗尽  

产业增强：并行子 Agent、网页浏览工具、表格/论文阅读器、人类复核节点、可复现研究日志。


In [ ]:
def should_stop(state: ResearchState, min_cover=1) -> bool:
    covered = {sq: 0 for sq in state.subquestions}
    for e in state.evidence:
        for sq in state.subquestions:
            if sq in e.query or any(t in e.snippet.lower() for t in sq.lower().split()[-2:]):
                covered[sq] += 1
    if all(v >= min_cover for v in covered.values()):
        return True
    if state.budget_search <= 0:
        return True
    return False


state.done = should_stop(state)
print('=== Stop Decision ===')
print('done=', state.done, 'remaining_budget=', state.budget_search)
print(f'\nKey: Deep research is a budgeted MDP; fluency without stop rules burns cost and still hallucinates.')


## 课后思考题

1. 深度研究与单次 RAG 的本质差别是什么？哪些产品问题其实不需要深度研究？
2. 如何度量“信息增益”以避免无意义循环检索？
3. 多 Agent 并行检索时如何合并证据并去重？
4. 引用了来源仍可能误读原文，如何做事实核对？

---
> 本节涵盖了12.7 深度研究智能体的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
